In [0]:
# =====================================================================================
# Notebook: 99_validation/01_bronze_checker.py
# Project : nba-datalakehouse
# Purpose : Validate Bronze Delta tables in Unity Catalog.
#
# Validates:
#   1) Tables exist in the correct catalog/schema
#   2) Basic row counts are non-zero
#   3) Partitioning by dt is working (daily loads visible)
#   4) Bronze contract is consistent: columns include (dt, meta, data)
#   5) Duplicates are not unexpectedly exploding (basic sanity)
#
# IMPORTANT:
#   - This notebook reads ONLY Delta tables via spark.table(...)
#   - It does NOT read raw landing JSON (prevents schema-merge errors)
# =====================================================================================

from pyspark.sql import functions as F

BRONZE_SCHEMA = "workspace.nba_bronze"

# (table_name, json_path_for_business_key_inside_data)
TABLES = [
    ("balldontlie_games",      "$.id"),
    ("balldontlie_teams",      "$.id"),
    ("the_odds_api_sports",    "$.key"),
    ("the_odds_api_events",    "$.id"),
    ("the_odds_api_odds",      "$.id"),
]

def fqn(name: str) -> str:
    return f"{BRONZE_SCHEMA}.{name}"

def table_exists(table_fqn: str) -> bool:
    try:
        spark.table(table_fqn)
        return True
    except Exception:
        return False

def missing_cols(df, required):
    cols = set(df.columns)
    return [c for c in required if c not in cols]

results = []

# -----------------------------
# 1) Tables exist
# -----------------------------
print("=== 1) Table existence checks ===")
for name, _ in TABLES:
    t = fqn(name)
    ok = table_exists(t)
    print(t, "exists=", ok)
    results.append((t, "exists", ok))

# -----------------------------
# 2) Row counts non-zero
# -----------------------------
print("\n=== 2) Row count checks ===")
for name, _ in TABLES:
    t = fqn(name)
    if not table_exists(t):
        print("SKIP missing:", t)
        continue
    n = spark.table(t).count()
    print(t, "rows=", n)
    results.append((t, "row_count", n))

# -----------------------------
# 3) dt distribution (latest 15)
# -----------------------------
print("\n=== 3) dt partition distribution (latest 15) ===")
for name, _ in TABLES:
    t = fqn(name)
    if not table_exists(t):
        continue
    df = spark.table(t)
    if "dt" not in df.columns:
        print("WARNING dt missing:", t)
        continue
    print("\n", t)
    (df.groupBy("dt")
       .count()
       .orderBy(F.col("dt").desc())
       .show(15, truncate=False))

# -----------------------------
# 4) Bronze contract consistency
# -----------------------------
print("\n=== 4) Bronze contract checks: (dt, meta, data) ===")
for name, _ in TABLES:
    t = fqn(name)
    if not table_exists(t):
        continue
    df = spark.table(t)
    missing = missing_cols(df, ["dt", "meta", "data"])
    if missing:
        print(t, "MISSING:", missing)
        results.append((t, "contract_ok", False))
    else:
        print(t, "OK: has dt, meta, data")
        results.append((t, "contract_ok", True))

# -----------------------------
# 5) Duplicate sanity (by id within dt)
# -----------------------------
print("\n=== 5) Duplicate sanity (business key within dt) ===")
DUP_WARN_THRESHOLD = 0

for name, json_path in TABLES:
    t = fqn(name)
    if not table_exists(t):
        continue

    df = spark.table(t)
    missing = missing_cols(df, ["dt", "data"])
    if missing:
        print("SKIP dup check (missing cols):", t, missing)
        continue

    df_ids = (
        df.select(
            F.col("dt"),
            F.get_json_object(F.col("data"), json_path).alias("_id"),
        )
        .filter(F.col("_id").isNotNull())
    )

    dup = (
        df_ids.groupBy("dt", "_id")
              .count()
              .filter(F.col("count") > 1)
    )

    dup_groups = dup.count()
    print(t, "duplicate_groups(dt,_id) =", dup_groups)
    results.append((t, "duplicate_groups", dup_groups))

    if dup_groups > DUP_WARN_THRESHOLD:
        dup.orderBy(F.col("count").desc()).show(20, truncate=False)

results_str = [(a, b, str(c)) for (a, b, c) in results]
display(results_str)


In [0]:
print("About to list bronze tables only (no landing reads).")
display(spark.sql("SHOW TABLES IN workspace.nba_bronze"))


In [0]:
from pyspark.sql import functions as F

tables = [
    "workspace.nba_bronze.balldontlie_games",
    "workspace.nba_bronze.balldontlie_teams",
    "workspace.nba_bronze.the_odds_api_sports",
    "workspace.nba_bronze.the_odds_api_events",
    "workspace.nba_bronze.the_odds_api_odds",
]

for t in tables:
    try:
        df = spark.table(t)
        print(t, "rows=", df.count(), "cols=", df.columns)
        df.groupBy("dt").count().orderBy(F.col("dt").desc()).show(5, truncate=False)
    except Exception as e:
        print("SKIP/ERROR:", t, "->", str(e)[:300])


In [0]:
from pyspark.sql import functions as F

tables = [
    "workspace.nba_bronze.balldontlie_games",
    "workspace.nba_bronze.balldontlie_teams",
    "workspace.nba_bronze.the_odds_api_sports",
    "workspace.nba_bronze.the_odds_api_events",
    "workspace.nba_bronze.the_odds_api_odds",
]

for t in tables:
    try:
        df = spark.table(t)
        print(t, "rows=", df.count(), "cols=", df.columns)
        df.groupBy("dt").count().orderBy(F.col("dt").desc()).show(5, truncate=False)
    except Exception as e:
        print("SKIP/ERROR:", t, "->", str(e)[:300])
